# Unseen audio depth evaluation

Runs inference on **completely unseen FLAC files** using a saved model from a v2 training run,
then produces structured metrics and publication-quality plots.

Supports both task modes:
- **Classification** — discrete depth-step prediction (0.1 – 1.0 mm)
- **Regression** — continuous depth prediction

Set `PREDICTION_MODE = 'load_csv'` to evaluate an already-saved prediction CSV, or
`'run_model'` to run the model live on a new folder.


In [ ]:
import importlib
import json
import re
import sys
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import MaxNLocator, PercentFormatter
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
)

# ── Global plot style ────────────────────────────────────────────────────────
plt.style.use("default")
plt.rcParams.update(
    {
        "figure.figsize": (9.0, 5.6),
        "figure.facecolor": "white",
        "axes.facecolor": "#FAFAFA",
        "axes.grid": True,
        "grid.alpha": 0.28,
        "grid.linestyle": "--",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.spines.left": True,
        "axes.spines.bottom": True,
        "axes.titleweight": "bold",
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "xtick.labelsize": 9.5,
        "ytick.labelsize": 9.5,
        "legend.frameon": False,
        "legend.fontsize": 10,
        "savefig.bbox": "tight",
        "savefig.dpi": 180,
        "font.family": "sans-serif",
    }
)

# ── Colour palette ───────────────────────────────────────────────────────────
PAL = {
    "blue": "#1f77b4",
    "orange": "#e07b39",
    "green": "#2ca02c",
    "red": "#d62728",
    "purple": "#7a4f9e",
    "grey": "#7f7f7f",
    "teal": "#17becf",
    "correct": "#2ca02c",
    "wrong": "#d62728",
    "primary": "#1f77b4",
    "secondary": "#e07b39",
}

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

## 1 · Settings

In [ ]:
# ── Project root ──────────────────────────────────────────────────────────────
PROJECT_DIR = Path("..").resolve()
if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = Path(".").resolve()

# ── Training output to evaluate ───────────────────────────────────────────────
# Point this to the folder produced by train.py, e.g.:
#   outputs/spec_resnet_reg   outputs/transformer_cls   outputs/small_cnn_reg
OUTPUT_DIR = PROJECT_DIR / "outputs/transformer_reg"

# Model label is derived automatically from the output folder name.
# Override here only if you want a custom label in saved file names.
MODEL_LABEL = OUTPUT_DIR.stem  # e.g. 'transformer_reg'

# ── How to obtain predictions ─────────────────────────────────────────────────
PREDICTION_MODE = "run_model"  # 'load_csv'  or  'run_model'

# Option A — load an existing CSV (load_csv mode)
PREDICTIONS_CSV = OUTPUT_DIR / "repeat_001" / "test_file_predictions.csv"

# Option B — run model live on a new, unseen folder (run_model mode)
UNSEEN_DATA_DIR = PROJECT_DIR / "data/inflated"
FILE_GLOB = "**/*.flac"
DEVICE_OVERRIDE = "auto"  # 'auto' | 'cpu' | 'cuda'
BATCH_SIZE_OVERRIDE = None  # None → use config value

# Predictions are saved here (run_model mode only)
OUTPUT_PREDICTIONS_CSV = OUTPUT_DIR / f"inference/{MODEL_LABEL}_predictions_inflated.csv"

# ── Evaluation settings ───────────────────────────────────────────────────────
DEFAULT_STEP_MM = 0.1
UNCERTAINTY_BAND_MM = 0.015  # regression: band around class boundaries
TOP_N_CASES = 20  # rows in worst-case tables
PREVIEW_N_ROWS = 5

SAVE_PLOTS = True
SAVE_TABLES = True

## 2 · Project integration helpers

In [ ]:
# ── Module loading ────────────────────────────────────────────────────────────


def load_project_modules(project_dir: Path):
    project_dir = Path(project_dir).resolve()
    src_dir = project_dir / "src"
    if not src_dir.exists():
        raise FileNotFoundError(f"src/ not found under {project_dir}")
    src_str = str(src_dir)
    if src_str not in sys.path:
        sys.path.insert(0, src_str)
    pkg = "depth_audio_baseline"
    mods = {
        name: importlib.import_module(f"{pkg}.{name}")
        for name in ("config", "data", "engine", "models", "utils")
    }
    return pkg, mods


def resolve_model_dir(output_dir: Path) -> Path:
    output_dir = Path(output_dir)
    final = output_dir / "final_model"
    if final.exists() and (final / "best_model.pt").exists():
        return final
    if (output_dir / "best_model.pt").exists():
        return output_dir
    raise FileNotFoundError(f"No best_model.pt found in {output_dir} or {output_dir}/final_model")


def load_cfg(model_dir: Path, mods, batch_size_override=None, device_override="auto"):
    cfg_path = Path(model_dir) / "config.json"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = mods["config"].TrainConfig.from_json_dict(json.load(f))
    if batch_size_override is not None:
        cfg.batch_size = int(batch_size_override)
    if device_override in {"cpu", "cuda"}:
        cfg.device = device_override
    return cfg


def normalize_class_to_depth(mapping: dict) -> dict[int, float]:
    direct, inverted = {}, {}
    for k, v in mapping.items():
        kn = pd.to_numeric(pd.Series([k]), errors="coerce").iloc[0]
        vn = pd.to_numeric(pd.Series([v]), errors="coerce").iloc[0]
        if pd.notna(kn) and pd.notna(vn):
            if np.isclose(kn, round(kn)):
                direct[int(round(kn))] = float(vn)
            if np.isclose(vn, round(vn)):
                inverted[int(round(vn))] = float(kn)
    result = direct if len(direct) >= len(inverted) else inverted
    return dict(sorted(result.items()))


def load_class_mapping(model_dir: Path, mods):
    raw = mods["utils"].read_label_mapping(Path(model_dir) / "label_mapping.json")
    class_to_depth = normalize_class_to_depth(raw)
    depth_to_class = {v: k for k, v in class_to_depth.items()}
    return depth_to_class, class_to_depth


# ── Inference ─────────────────────────────────────────────────────────────────


def run_project_inference(
    project_dir,
    output_dir,
    unseen_data_dir,
    file_glob="**/*.flac",
    device_override="auto",
    batch_size_override=None,
    output_predictions_csv=None,
):
    import torch

    pkg, mods = load_project_modules(project_dir)
    model_dir = resolve_model_dir(output_dir)
    cfg = load_cfg(
        model_dir, mods, batch_size_override=batch_size_override, device_override=device_override
    )
    cfg.data_dir = str(unseen_data_dir)
    cfg.file_glob = file_glob
    device = mods["utils"].choose_device(cfg.device)

    print(f"Model dir : {model_dir}")
    print(f"Task      : {cfg.task}  |  Feature: {cfg.feature_type}  |  Device: {device}")

    file_df = mods["utils"].build_file_table(unseen_data_dir, cfg.file_glob)
    file_df = mods["utils"].attach_step_idx_if_possible(file_df)
    file_df["split_group_id"] = file_df["file_group_id"].astype(str)
    file_df["split"] = "infer"

    if cfg.task == "classification":
        depth_to_class, class_to_depth = load_class_mapping(output_dir, mods)
        file_df["class_idx"] = file_df["depth_mm"].map(depth_to_class)
        if file_df["class_idx"].isna().any():
            raise ValueError("Some depths in unseen data are absent from the training mapping.")
        file_df["class_idx"] = file_df["class_idx"].astype(int)
        out_dim = len(class_to_depth)
    else:
        file_df, _, class_to_depth = mods["utils"].add_class_labels(file_df)
        out_dim = 1

    ds = mods["data"].WaveformWindowDataset(file_df=file_df, cfg=cfg, training=False)
    loader = mods["engine"].make_loader(ds, cfg, shuffle=False)

    model = mods["models"].DepthModel(cfg, out_dim=out_dim).to(device)
    model.load_state_dict(torch.load(model_dir / "best_model.pt", map_location=device))
    model.eval()

    pred_window_df = mods["engine"].predict_loader(model, loader, device, cfg)
    pred_file_df = mods["engine"].aggregate_file_predictions(
        pred_window_df, file_df, cfg, class_to_depth
    )

    if output_predictions_csv is not None:
        p = Path(output_predictions_csv)
        p.parent.mkdir(parents=True, exist_ok=True)
        pred_file_df.to_csv(p, index=False)
        print(f"Predictions saved → {p}")

    return pred_file_df


# ── Standardise column names ──────────────────────────────────────────────────


def _extract_depth_from_name(row: pd.Series) -> float:
    for col in ("name", "path", "recording_root"):
        if col not in row.index or pd.isna(row[col]):
            continue
        m = re.search(r"depth([0-9]+(?:\.[0-9]+)?)", str(row[col]), re.IGNORECASE)
        if m:
            return float(m.group(1))
    return np.nan


def standardize_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    renames = {
        "y_true_depth": "true_depth_mm",
        "y_pred_depth": "pred_depth_mm",
        "y_true_class": "true_class",
        "y_pred_class": "pred_class",
    }
    for old, new in renames.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})
    if "true_depth_mm" not in df.columns and "depth_mm" in df.columns:
        df["true_depth_mm"] = df["depth_mm"]
    if "true_depth_mm" not in df.columns:
        df["true_depth_mm"] = df.apply(_extract_depth_from_name, axis=1)

    prob_cols = sorted(
        [c for c in df.columns if str(c).startswith("p_")],
        key=lambda x: int(str(x).split("_", 1)[1]),
    )
    if prob_cols and "confidence_top1" not in df.columns:
        probs = df[prob_cols].to_numpy(dtype=float)
        s = np.sort(probs, axis=1)
        df["confidence_top1"] = s[:, -1]
        df["confidence_gap"] = (s[:, -1] - s[:, -2]) if s.shape[1] >= 2 else s[:, -1]

    if "run_id" not in df.columns:
        if "recording_root" in df.columns:
            df["run_id"] = df["recording_root"].astype(str)
        elif "name" in df.columns:
            df["run_id"] = df["name"].astype(str).str.split("__seg").str[0]
        else:
            df["run_id"] = "unknown"

    for req in ("true_depth_mm", "pred_depth_mm"):
        if req not in df.columns:
            raise ValueError(f"Required column missing after standardisation: {req}")
    return df


def detect_task(df: pd.DataFrame) -> str:
    if "pred_class" in df.columns or any(str(c).startswith("p_") for c in df.columns):
        return "classification"
    return "regression"


def make_eval_dirs(output_dir: Path, task: str, model_label: str):
    base = Path(output_dir) / "evaluation" / f"{model_label}_{task}"
    plots = base / "plots"
    tables = base / "tables"
    plots.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return base, plots, tables


def load_or_create_predictions() -> pd.DataFrame:
    if PREDICTION_MODE == "load_csv":
        df = pd.read_csv(PREDICTIONS_CSV)
        print(f"Loaded predictions: {PREDICTIONS_CSV}")
    elif PREDICTION_MODE == "run_model":
        df = run_project_inference(
            project_dir=PROJECT_DIR,
            output_dir=OUTPUT_DIR,
            unseen_data_dir=UNSEEN_DATA_DIR,
            file_glob=FILE_GLOB,
            device_override=DEVICE_OVERRIDE,
            batch_size_override=BATCH_SIZE_OVERRIDE,
            output_predictions_csv=OUTPUT_PREDICTIONS_CSV,
        )
    else:
        raise ValueError("PREDICTION_MODE must be 'load_csv' or 'run_model'")
    return standardize_df(df)

## 3 · Load predictions

In [ ]:
df = load_or_create_predictions()
TASK = detect_task(df)

EVAL_DIR, PLOTS_DIR, TABLES_DIR = make_eval_dirs(OUTPUT_DIR, TASK, MODEL_LABEL)

print(f"Task        : {TASK}")
print(f"Files       : {len(df)}")
print(f"Output base : {EVAL_DIR}")
print()

_preview_cols = [
    c
    for c in [
        "name",
        "run_id",
        "true_depth_mm",
        "pred_depth_mm",
        "true_class",
        "pred_class",
        "confidence_top1",
    ]
    if c in df.columns
]
display(df[_preview_cols].head(PREVIEW_N_ROWS))

## 4 · Evaluation tables

In [ ]:
# ── Shared save helpers ───────────────────────────────────────────────────────


def save_table(df: pd.DataFrame, path: Path):
    if SAVE_TABLES:
        df.to_csv(path, index=False)
        print(f"  Saved table → {path.name}")


def save_summary_json(df: pd.DataFrame, path: Path):
    if SAVE_TABLES:
        payload = {}
        for _, row in df.iterrows():
            for col in df.columns:
                try:
                    payload[str(col)] = float(row[col])
                except Exception:
                    payload[str(col)] = str(row[col])
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        print(f"  Saved JSON  → {path.name}")


# ── Feature engineering helpers ───────────────────────────────────────────────


def infer_step_mm(df: pd.DataFrame, default: float = 0.1) -> float:
    vals = np.sort(df["true_depth_mm"].dropna().astype(float).unique())
    diffs = np.diff(vals)
    diffs = diffs[diffs > 1e-9]
    return float(np.round(np.median(diffs), 6)) if diffs.size else default


def add_classification_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["true_class"] = out["true_class"].astype(int)
    out["pred_class"] = out["pred_class"].astype(int)
    out["correct"] = out["true_class"] == out["pred_class"]
    out.attrs["labels"] = sorted(out["true_class"].unique().tolist())
    prob_cols = sorted(
        [c for c in out.columns if str(c).startswith("p_")],
        key=lambda x: int(str(x).split("_", 1)[1]),
    )
    out.attrs["prob_cols"] = prob_cols
    return out


def add_regression_cols(
    df: pd.DataFrame, step_mm: float, uncertainty_band_mm: float
) -> pd.DataFrame:
    out = df.copy()
    out["signed_error"] = out["pred_depth_mm"] - out["true_depth_mm"]
    out["abs_error"] = out["signed_error"].abs()
    out["sq_error"] = out["signed_error"] ** 2

    levels = sorted(out["true_depth_mm"].dropna().astype(float).unique())
    boundaries = [(levels[i] + levels[i + 1]) / 2 for i in range(len(levels) - 1)]

    if boundaries:
        ba = np.array(boundaries)
        dtb = np.min(np.abs(out["pred_depth_mm"].to_numpy()[:, None] - ba[None, :]), axis=1)
    else:
        dtb = np.full(len(out), np.inf)

    half = max(step_mm / 2, 1e-9)
    out["dist_to_boundary"] = dtb
    out["certainty"] = np.clip(dtb / half, 0.0, 1.0)
    out["is_confident"] = dtb >= float(uncertainty_band_mm)

    out.attrs["levels"] = levels
    out.attrs["boundaries"] = boundaries
    out.attrs["step_mm"] = step_mm
    out.attrs["uncertainty_band_mm"] = uncertainty_band_mm
    return out


# ── Summary builders ─────────────────────────────────────────────────────────


def summarize_classification(df: pd.DataFrame) -> pd.DataFrame:
    y_true = df["true_class"].to_numpy()
    y_pred = df["pred_class"].to_numpy()
    row = {
        "n_files": len(df),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "n_errors": int((y_true != y_pred).sum()),
        "error_rate": float((y_true != y_pred).mean()),
    }
    if "confidence_top1" in df.columns:
        row["mean_confidence_correct"] = float(df.loc[df["correct"], "confidence_top1"].mean())
        row["mean_confidence_wrong"] = float(df.loc[~df["correct"], "confidence_top1"].mean())
    return pd.DataFrame([row])


def per_class_summary(df: pd.DataFrame) -> pd.DataFrame:
    y_true = df["true_class"].to_numpy()
    y_pred = df["pred_class"].to_numpy()
    labels = df.attrs.get("labels", sorted(np.unique(y_true).tolist()))
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )
    rows = []
    for i, lab in enumerate(labels):
        depth = df.loc[df["true_class"] == lab, "true_depth_mm"].median()
        rows.append(
            {
                "class_idx": lab,
                "depth_mm": round(float(depth), 3) if pd.notna(depth) else np.nan,
                "n_files": int(sup[i]),
                "precision": round(float(prec[i]), 4),
                "recall": round(float(rec[i]), 4),
                "f1": round(float(f1[i]), 4),
            }
        )
    return pd.DataFrame(rows)


def per_run_classification_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for run_id, g in df.groupby("run_id"):
        y_true = g["true_class"].to_numpy()
        y_pred = g["pred_class"].to_numpy()
        rows.append(
            {
                "run_id": run_id,
                "n_files": len(g),
                "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
                "macro_f1": round(
                    float(f1_score(y_true, y_pred, average="macro", zero_division=0)), 4
                ),
                "n_errors": int((y_true != y_pred).sum()),
            }
        )
    return pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)


def classification_cases(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    errors = df.loc[~df["correct"]].copy()
    cols = [
        c
        for c in [
            "name",
            "run_id",
            "true_depth_mm",
            "pred_depth_mm",
            "true_class",
            "pred_class",
            "confidence_top1",
        ]
        if c in errors.columns
    ]
    if "confidence_top1" in errors.columns:
        errors = errors.sort_values("confidence_top1", ascending=False)
    return errors[cols].head(top_n).reset_index(drop=True)


def summarize_regression(df: pd.DataFrame) -> pd.DataFrame:
    y_true = df["true_depth_mm"].to_numpy()
    y_pred = df["pred_depth_mm"].to_numpy()
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    bias = float(np.mean(y_pred - y_true))
    p90 = float(np.percentile(np.abs(y_pred - y_true), 90))

    step_mm = df.attrs.get("step_mm", DEFAULT_STEP_MM)
    step_acc = float(np.mean(np.round(y_true / step_mm) == np.round(y_pred / step_mm)))

    conf_pct = float(df["is_confident"].mean()) if "is_confident" in df.columns else np.nan
    conf_mae = (
        float(df.loc[df["is_confident"], "abs_error"].mean())
        if "is_confident" in df.columns and df["is_confident"].any()
        else np.nan
    )

    row = {
        "n_files": len(df),
        "mae_mm": round(mae, 5),
        "rmse_mm": round(rmse, 5),
        "r2": round(r2, 4),
        "bias_mm": round(bias, 5),
        "p90_abs_error_mm": round(p90, 5),
        "step_accuracy": round(step_acc, 4),
        "confident_pct": round(conf_pct, 4),
        "mae_confident_mm": round(conf_mae, 5) if pd.notna(conf_mae) else np.nan,
    }
    return pd.DataFrame([row])


def per_depth_regression_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for depth, g in df.groupby("true_depth_mm"):
        rows.append(
            {
                "true_depth_mm": round(float(depth), 3),
                "n_files": len(g),
                "mae_mm": round(float(g["abs_error"].mean()), 5),
                "rmse_mm": round(float(np.sqrt((g["sq_error"]).mean())), 5),
                "bias_mm": round(float(g["signed_error"].mean()), 5),
                "p90_ae_mm": round(float(g["abs_error"].quantile(0.90)), 5),
            }
        )
    return pd.DataFrame(rows).sort_values("true_depth_mm").reset_index(drop=True)


def per_run_regression_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for run_id, g in df.groupby("run_id"):
        rows.append(
            {
                "run_id": run_id,
                "n_files": len(g),
                "mae_mm": round(float(g["abs_error"].mean()), 5),
                "rmse_mm": round(float(np.sqrt(g["sq_error"].mean())), 5),
                "bias_mm": round(float(g["signed_error"].mean()), 5),
            }
        )
    return pd.DataFrame(rows).sort_values("mae_mm").reset_index(drop=True)


def regression_cases(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    cols = [
        c
        for c in [
            "name",
            "run_id",
            "true_depth_mm",
            "pred_depth_mm",
            "signed_error",
            "abs_error",
            "certainty",
        ]
        if c in df.columns
    ]
    return df.nlargest(top_n, "abs_error")[cols].reset_index(drop=True)


# ── Execute ───────────────────────────────────────────────────────────────────

step_mm = infer_step_mm(df, DEFAULT_STEP_MM)

if TASK == "classification":
    eval_df = add_classification_cols(df)
    sum_df = summarize_classification(eval_df)
    class_df = per_class_summary(eval_df)
    run_df = per_run_classification_summary(eval_df)
    cases_df = classification_cases(eval_df, TOP_N_CASES)

    print("── Overall ──────────────────────────────────────────────────")
    display(sum_df)
    print("── Per class ────────────────────────────────────────────────")
    display(class_df)
    print("── Per recording session ────────────────────────────────────")
    display(run_df)
    print("── High-confidence misclassifications ───────────────────────")
    display(cases_df)

    if SAVE_TABLES:
        save_table(sum_df, TABLES_DIR / "cls_overall.csv")
        save_table(class_df, TABLES_DIR / "cls_per_class.csv")
        save_table(run_df, TABLES_DIR / "cls_per_run.csv")
        save_table(cases_df, TABLES_DIR / "cls_key_errors.csv")
        save_summary_json(sum_df, TABLES_DIR / "cls_overall.json")

else:
    eval_df = add_regression_cols(df, step_mm, UNCERTAINTY_BAND_MM)
    sum_df = summarize_regression(eval_df)
    depth_df = per_depth_regression_summary(eval_df)
    run_df = per_run_regression_summary(eval_df)
    cases_df = regression_cases(eval_df, TOP_N_CASES)

    print("── Overall ──────────────────────────────────────────────────")
    display(sum_df)
    print("── Per depth level ──────────────────────────────────────────")
    display(depth_df)
    print("── Per recording session ────────────────────────────────────")
    display(run_df)
    print("── Largest errors ───────────────────────────────────────────")
    display(cases_df)

    if SAVE_TABLES:
        save_table(sum_df, TABLES_DIR / "reg_overall.csv")
        save_table(depth_df, TABLES_DIR / "reg_per_depth.csv")
        save_table(run_df, TABLES_DIR / "reg_per_run.csv")
        save_table(cases_df, TABLES_DIR / "reg_worst_cases.csv")
        save_summary_json(sum_df, TABLES_DIR / "reg_overall.json")

## 5 · Plots

In [ ]:
def savefig(fig, name: str):
    if SAVE_PLOTS:
        p = PLOTS_DIR / name
        fig.savefig(p)
        print(f"  Saved → {p.name}")
    plt.show()
    plt.close(fig)


# ── Depth-level colour map (shared across regression plots) ───────────────────


def _depth_cmap(levels):
    """Returns a dict mapping each depth level to a colour from a blue gradient."""
    cmap = plt.get_cmap("Blues_r" if len(levels) > 1 else "Blues")
    norm = mcolors.Normalize(vmin=0, vmax=max(len(levels) - 1, 1))
    return {d: cmap(norm(i)) for i, d in enumerate(sorted(levels))}


# ════════════════════════════════════════════════════════════════════════════
# CLASSIFICATION PLOTS
# ════════════════════════════════════════════════════════════════════════════


def plot_cls_confusion_matrix(df: pd.DataFrame):
    labels = df.attrs.get("labels", sorted(df["true_class"].unique()))
    # depth labels for axes
    depth_map = {row["class_idx"]: row["depth_mm"] for _, row in per_class_summary(df).iterrows()}
    tick_lbl = [f"{depth_map.get(l, l):.1f}" for l in labels]

    cm = confusion_matrix(df["true_class"], df["pred_class"], labels=labels)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(7.6, 6.4))
    im = ax.imshow(cm_norm, vmin=0, vmax=1, cmap="Blues", aspect="auto")
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Row-normalised fraction", fontsize=10)

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(tick_lbl, rotation=45, ha="right")
    ax.set_yticklabels(tick_lbl)
    ax.set_xlabel("Predicted depth [mm]")
    ax.set_ylabel("True depth [mm]")
    ax.set_title("Confusion matrix")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            pct = cm_norm[i, j]
            color = "white" if pct > 0.55 else "#222222"
            ax.text(
                j,
                i,
                f"{pct:.0%}\n({cm[i, j]})",
                ha="center",
                va="center",
                color=color,
                fontsize=8.5,
                fontweight="bold" if i == j else "normal",
            )

    fig.tight_layout()
    savefig(fig, "cls_confusion_matrix.png")


def plot_cls_per_class_metrics(class_df: pd.DataFrame):
    df_plot = class_df.sort_values("depth_mm").reset_index(drop=True)
    labels = [f"{r['depth_mm']:.1f}" for _, r in df_plot.iterrows()]
    y = np.arange(len(df_plot))
    w = 0.26

    fig, ax = plt.subplots(figsize=(8.8, max(5.0, 0.52 * len(df_plot) + 1.6)))
    ax.barh(y + w, df_plot["precision"], w, label="Precision", color=PAL["blue"], alpha=0.88)
    ax.barh(y, df_plot["recall"], w, label="Recall", color=PAL["orange"], alpha=0.88)
    ax.barh(y - w, df_plot["f1"], w, label="F1", color=PAL["green"], alpha=0.88)

    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlim(0, 1.08)
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    ax.set_xlabel("Score")
    ax.set_ylabel("Depth [mm]")
    ax.set_title("Per-class precision, recall and F1")
    ax.legend(loc="lower right", ncol=3, fontsize=10)

    # annotate F1 values
    for i, (_, row) in enumerate(df_plot.iterrows()):
        ax.text(
            row["f1"] + 0.01,
            y[i] - w,
            f"{row['f1']:.2f}",
            va="center",
            fontsize=8.5,
            color=PAL["green"],
        )

    fig.tight_layout()
    savefig(fig, "cls_per_class_metrics.png")


def plot_cls_per_run_accuracy(run_df: pd.DataFrame):
    if run_df.empty:
        return
    df_plot = run_df.sort_values("accuracy", ascending=True).reset_index(drop=True)

    norm = mcolors.Normalize(vmin=df_plot["accuracy"].min() - 0.05, vmax=1.0)
    colors = [plt.get_cmap("RdYlGn")(norm(v)) for v in df_plot["accuracy"]]

    fig, ax = plt.subplots(figsize=(9.2, max(4.0, 0.48 * len(df_plot) + 1.4)))
    bars = ax.barh(
        df_plot["run_id"], df_plot["accuracy"], color=colors, edgecolor="white", linewidth=0.6
    )
    for bar, val in zip(bars, df_plot["accuracy"]):
        ax.text(
            min(val + 0.005, 0.99),
            bar.get_y() + bar.get_height() / 2,
            f"{val:.1%}",
            va="center",
            fontsize=9,
        )
    ax.set_xlim(0, 1.0)
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    ax.set_xlabel("Accuracy")
    ax.set_ylabel("Recording session")
    ax.set_title("Per-session classification accuracy")
    ax.axvline(
        df_plot["accuracy"].mean(),
        linestyle="--",
        color=PAL["grey"],
        linewidth=1.2,
        label=f"Mean {df_plot['accuracy'].mean():.1%}",
    )
    ax.legend(loc="lower right")
    fig.tight_layout()
    savefig(fig, "cls_per_run_accuracy.png")


def plot_cls_confidence_distribution(df: pd.DataFrame):
    if "confidence_top1" not in df.columns:
        print("No confidence_top1 column — skipping confidence plot.")
        return
    correct = df.loc[df["correct"], "confidence_top1"]
    wrong = df.loc[~df["correct"], "confidence_top1"]
    bins = np.linspace(0, 1, 18)

    fig, ax = plt.subplots(figsize=(8.4, 4.8))
    ax.hist(
        correct,
        bins=bins,
        alpha=0.78,
        color=PAL["correct"],
        label=f"Correct  (n={len(correct)})",
        edgecolor="white",
        linewidth=0.6,
    )
    ax.hist(
        wrong,
        bins=bins,
        alpha=0.78,
        color=PAL["wrong"],
        label=f"Wrong    (n={len(wrong)})",
        edgecolor="white",
        linewidth=0.6,
    )
    ax.axvline(
        correct.mean() if len(correct) else 0, linestyle="--", color=PAL["correct"], linewidth=1.6
    )
    ax.axvline(wrong.mean() if len(wrong) else 0, linestyle="--", color=PAL["wrong"], linewidth=1.6)
    ax.set_xlabel("Top-1 softmax confidence")
    ax.set_ylabel("Number of files")
    ax.set_title("Classification confidence distribution")
    ax.legend(loc="upper left")
    fig.tight_layout()
    savefig(fig, "cls_confidence_distribution.png")


# ════════════════════════════════════════════════════════════════════════════
# REGRESSION PLOTS
# ════════════════════════════════════════════════════════════════════════════


def plot_reg_true_vs_pred(df: pd.DataFrame, sum_df: pd.DataFrame):
    mn = float(min(df["true_depth_mm"].min(), df["pred_depth_mm"].min()))
    mx = float(max(df["true_depth_mm"].max(), df["pred_depth_mm"].max()))
    pad = 0.04
    lo, hi = mn - pad, mx + pad
    xx = np.linspace(lo, hi, 300)

    fig, ax = plt.subplots(figsize=(7.2, 6.4))
    sc = ax.scatter(
        df["true_depth_mm"],
        df["pred_depth_mm"],
        c=df["abs_error"],
        cmap="RdYlGn_r",
        s=62,
        alpha=0.88,
        edgecolor="white",
        linewidth=0.5,
        vmin=0,
        vmax=df["abs_error"].quantile(0.95),
    )
    cb = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Absolute error [mm]", fontsize=10)

    ax.plot(xx, xx, "--", linewidth=1.6, color="#333333", label="Perfect prediction")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel("True depth [mm]")
    ax.set_ylabel("Predicted depth [mm]")
    ax.set_title("True vs predicted depth")

    mae = float(sum_df["mae_mm"].iloc[0])
    r2 = float(sum_df["r2"].iloc[0])
    ax.text(
        0.04,
        0.96,
        f"MAE = {mae:.4f} mm\nR² = {r2:.4f}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#cccccc", alpha=0.92),
    )
    ax.legend(loc="lower right")
    ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    savefig(fig, "reg_true_vs_pred.png")


def plot_reg_error_by_depth(df: pd.DataFrame):
    levels = sorted(df["true_depth_mm"].unique())
    grouped = [df.loc[df["true_depth_mm"] == d, "signed_error"].to_numpy() for d in levels]
    colors = list(_depth_cmap(levels).values())

    fig, ax = plt.subplots(figsize=(9.6, 5.2))
    bp = ax.boxplot(
        grouped,
        tick_labels=[f"{d:.1f}" for d in levels],
        patch_artist=True,
        showfliers=True,
        flierprops=dict(marker="o", markersize=4, markerfacecolor=PAL["grey"], alpha=0.55),
    )
    for patch, col in zip(bp["boxes"], colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.80)
    for median in bp["medians"]:
        median.set_color("#333333")
        median.set_linewidth(1.8)
    for whisker in bp["whiskers"]:
        whisker.set_linewidth(1.1)

    ax.axhline(0, linestyle="--", linewidth=1.3, color="#333333")
    ax.set_xlabel("True depth [mm]")
    ax.set_ylabel("Signed error [mm]")
    ax.set_title("Signed prediction error by depth level")
    fig.tight_layout()
    savefig(fig, "reg_error_by_depth.png")


def plot_reg_per_run_mae(run_df: pd.DataFrame):
    if run_df.empty:
        return
    df_plot = run_df.sort_values("mae_mm", ascending=True).reset_index(drop=True)

    norm = mcolors.Normalize(vmin=0, vmax=df_plot["mae_mm"].max() * 1.1)
    colors = [plt.get_cmap("RdYlGn_r")(norm(v)) for v in df_plot["mae_mm"]]

    fig, ax = plt.subplots(figsize=(9.2, max(4.0, 0.48 * len(df_plot) + 1.4)))
    bars = ax.barh(
        df_plot["run_id"], df_plot["mae_mm"], color=colors, edgecolor="white", linewidth=0.6
    )
    for bar, val in zip(bars, df_plot["mae_mm"]):
        ax.text(
            val + df_plot["mae_mm"].max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}",
            va="center",
            fontsize=9,
        )
    ax.axvline(
        df_plot["mae_mm"].mean(),
        linestyle="--",
        color=PAL["grey"],
        linewidth=1.2,
        label=f"Mean {df_plot['mae_mm'].mean():.4f} mm",
    )
    ax.set_xlabel("MAE [mm]")
    ax.set_ylabel("Recording session")
    ax.set_title("Per-session regression MAE")
    ax.legend(loc="lower right")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    fig.tight_layout()
    savefig(fig, "reg_per_run_mae.png")


def plot_reg_abs_error_distribution(df: pd.DataFrame, sum_df: pd.DataFrame):
    mae = float(sum_df["mae_mm"].iloc[0])
    p90 = float(sum_df["p90_abs_error_mm"].iloc[0])
    median = float(df["abs_error"].median())

    fig, ax = plt.subplots(figsize=(8.6, 5.0))
    ax.hist(
        df["abs_error"], bins=16, color=PAL["blue"], alpha=0.80, edgecolor="white", linewidth=0.6
    )
    ax.axvline(
        median, linestyle="-", linewidth=2.0, color=PAL["primary"], label=f"Median  {median:.4f} mm"
    )
    ax.axvline(
        mae, linestyle="--", linewidth=2.0, color=PAL["orange"], label=f"Mean    {mae:.4f} mm"
    )
    ax.axvline(p90, linestyle=":", linewidth=2.0, color=PAL["red"], label=f"P90     {p90:.4f} mm")
    ax.set_xlabel("Absolute error [mm]")
    ax.set_ylabel("Number of files")
    ax.set_title("Absolute error distribution")
    ax.legend(loc="upper right")
    fig.tight_layout()
    savefig(fig, "reg_abs_error_distribution.png")


def plot_reg_per_depth_mae(depth_df: pd.DataFrame):
    df_plot = depth_df.sort_values("true_depth_mm").reset_index(drop=True)
    levels = df_plot["true_depth_mm"].tolist()
    colors = list(_depth_cmap(levels).values())

    fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.2), sharey=False)

    # MAE per depth
    ax = axes[0]
    bars = ax.bar(
        [f"{d:.1f}" for d in levels],
        df_plot["mae_mm"],
        color=colors,
        edgecolor="white",
        linewidth=0.7,
        alpha=0.88,
    )
    ax.axhline(
        df_plot["mae_mm"].mean(),
        linestyle="--",
        linewidth=1.4,
        color=PAL["grey"],
        label=f"Mean {df_plot['mae_mm'].mean():.4f} mm",
    )
    for bar, val in zip(bars, df_plot["mae_mm"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + df_plot["mae_mm"].max() * 0.01,
            f"{val:.4f}",
            ha="center",
            va="bottom",
            fontsize=8.5,
        )
    ax.set_xlabel("True depth [mm]")
    ax.set_ylabel("MAE [mm]")
    ax.set_title("MAE per depth level")
    ax.legend(fontsize=9)

    # Bias per depth
    ax = axes[1]
    bar_colors = [PAL["red"] if b > 0 else PAL["blue"] for b in df_plot["bias_mm"]]
    bars = ax.bar(
        [f"{d:.1f}" for d in levels],
        df_plot["bias_mm"],
        color=bar_colors,
        edgecolor="white",
        linewidth=0.7,
        alpha=0.82,
    )
    ax.axhline(0, linestyle="-", linewidth=1.0, color="#333333")
    ax.axhline(
        df_plot["bias_mm"].mean(),
        linestyle="--",
        linewidth=1.3,
        color=PAL["grey"],
        label=f"Mean bias {df_plot['bias_mm'].mean():+.4f} mm",
    )
    ax.set_xlabel("True depth [mm]")
    ax.set_ylabel("Bias [mm]  (pred − true)")
    ax.set_title("Systematic bias per depth level")
    ax.legend(fontsize=9)

    fig.suptitle("Regression error breakdown by depth level", fontweight="bold", y=1.01)
    fig.tight_layout()
    savefig(fig, "reg_per_depth_error.png")


def plot_reg_uncertainty_windows(df: pd.DataFrame):
    boundaries = df.attrs.get("boundaries", [])
    band = df.attrs.get("uncertainty_band_mm", UNCERTAINTY_BAND_MM)

    fig, ax = plt.subplots(figsize=(10.0, 5.2))

    for b in boundaries:
        ax.axvspan(b - band, b + band, alpha=0.14, color=PAL["red"], zorder=0)
        ax.axvline(b, color=PAL["red"], linewidth=0.8, alpha=0.45, zorder=1)

    sc = ax.scatter(
        df["pred_depth_mm"],
        df["abs_error"],
        c=df["certainty"],
        cmap="RdYlGn",
        s=58,
        alpha=0.88,
        edgecolor="white",
        linewidth=0.5,
        vmin=0,
        vmax=1,
        zorder=2,
    )
    cb = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label("Certainty score  (distance to boundary)", fontsize=10)

    ax.set_xlabel("Predicted depth [mm]")
    ax.set_ylabel("Absolute error [mm]")
    ax.set_title("Prediction error vs depth-step boundaries")
    ax.text(
        0.01,
        0.98,
        f"Red bands = ±{band:.3f} mm around step boundaries",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#cccccc", alpha=0.90),
    )
    fig.tight_layout()
    savefig(fig, "reg_uncertainty_windows.png")


# ── Execute plots ─────────────────────────────────────────────────────────────

if TASK == "classification":
    print("── Classification plots ─────────────────────────────────────")
    plot_cls_confusion_matrix(eval_df)
    plot_cls_per_class_metrics(class_df)
    plot_cls_per_run_accuracy(run_df)
    plot_cls_confidence_distribution(eval_df)
else:
    print("── Regression plots ─────────────────────────────────────────")
    plot_reg_true_vs_pred(eval_df, sum_df)
    plot_reg_error_by_depth(eval_df)
    plot_reg_abs_error_distribution(eval_df, sum_df)
    plot_reg_per_depth_mae(depth_df)
    plot_reg_per_run_mae(run_df)
    plot_reg_uncertainty_windows(eval_df)

## 6 · Summary

In [ ]:
print(f"Model   : {MODEL_LABEL}")
print(f"Task    : {TASK}")
print(f"Files   : {len(eval_df)}")
print()
if SAVE_TABLES:
    print(f"Tables → {TABLES_DIR}")
if SAVE_PLOTS:
    print(f"Plots  → {PLOTS_DIR}")
print()

if TASK == "classification":
    print("── Final metrics ────────────────────────────────────────────")
    display(
        sum_df[
            ["accuracy", "balanced_accuracy", "macro_f1", "n_errors", "error_rate"]
        ].style.format("{:.4f}")
    )
    print()
    print("── High-confidence misclassifications ───────────────────────")
    display(cases_df)
else:
    print("── Final metrics ────────────────────────────────────────────")
    display(
        sum_df[
            ["mae_mm", "rmse_mm", "r2", "bias_mm", "p90_abs_error_mm", "step_accuracy"]
        ].style.format("{:.5f}")
    )
    print()
    print("── Worst predictions ────────────────────────────────────────")
    display(cases_df)